# Advanced Problems: Nonlocal Scopes

These problems focus on nested functions, enclosing scopes, `nonlocal`, `global`, shadowing, and closure behavior.

**Instructions:** Predict the result first, then run the solution cells.

## Problem 1: Reading from an Enclosing Scope

Predict the output.

In [1]:
def outer():
    x = 'outer value'

    def inner():
        print(x)

    inner()

outer()

outer value


### Solution 1

Output:

```text
outer value
```

`inner()` can read `x` from the enclosing scope of `outer()` because Python searches local, enclosing, global, and built-in scopes.

---

## Problem 2: Local Shadowing

Predict the output.

In [2]:
def outer():
    x = 'outer'

    def inner():
        x = 'inner'
        print('inner:', x)

    inner()
    print('outer:', x)

outer()

inner: inner
outer: outer


### Solution 2

Output:

```text
inner: inner
outer: outer
```

`x = 'inner'` creates a new local variable inside `inner()`. It does not modify the `x` from `outer()`.

---

## Problem 3: Updating an Enclosing Variable

Fix the function so the inner function modifies the `x` from `outer()`.

In [3]:
def outer():
    x = 'outer'

    def inner():
        # Your fix here
        x = 'updated'

    inner()
    print(x)

# outer()

### Solution 3

In [4]:
def outer():
    x = 'outer'

    def inner():
        nonlocal x
        x = 'updated'

    inner()
    print(x)

outer()

updated


Expected output:

```text
updated
```

`nonlocal x` tells Python that `x` should be rebound in the nearest enclosing function scope where `x` exists.

---

## Problem 4: `nonlocal` Chooses the Nearest Enclosing Binding

Predict the output carefully.

In [5]:
def outer():
    x = 'outer'

    def middle():
        x = 'middle'

        def inner():
            nonlocal x
            x = 'inner updated'

        inner()
        print('middle:', x)

    middle()
    print('outer:', x)

outer()

middle: inner updated
outer: outer


### Solution 4

Output:

```text
middle: inner updated
outer: outer
```

`nonlocal x` inside `inner()` modifies the nearest enclosing `x`, which is the `x` inside `middle()`, not the one inside `outer()`.

---

## Problem 5: Skipping a Scope by Making It Nonlocal Too

Modify the code so that `inner()` ultimately changes `outer()`'s `x`.

In [6]:
def outer():
    x = 'outer'

    def middle():
        x = 'middle'

        def inner():
            nonlocal x
            x = 'changed'

        inner()

    middle()
    print(x)

# outer()

### Solution 5

In [7]:
def outer():
    x = 'outer'

    def middle():
        nonlocal x
        x = 'middle'

        def inner():
            nonlocal x
            x = 'changed'

        inner()

    middle()
    print(x)

outer()

changed


Expected output:

```text
changed
```

Because `middle()` also declares `nonlocal x`, it does not create its own local `x`. Both `middle()` and `inner()` refer to `outer()`'s `x`.

---

## Problem 6: `nonlocal` Cannot Target Globals

Predict what happens.

In [8]:
code = """
x = 100

def outer():
    def inner():
        nonlocal x
        x = 200
    inner()
"""

try:
    exec(code)
except Exception as ex:
    print(type(ex).__name__ + ':', ex)

SyntaxError: no binding for nonlocal 'x' found (<string>, line 6)


### Solution 6

This raises a `SyntaxError`:

```text
SyntaxError: no binding for nonlocal 'x' found
```

`nonlocal` only searches enclosing function scopes. It does not search the global/module scope.

---

## Problem 7: `global` and `nonlocal` Together

Predict the output.

In [9]:
x = 'global'

def outer():
    x = 'outer'

    def inner_nonlocal():
        nonlocal x
        x = 'changed outer'

    def inner_global():
        global x
        x = 'changed global'

    inner_nonlocal()
    inner_global()
    print('inside outer:', x)

outer()
print('module:', x)

inside outer: changed outer
module: changed global


### Solution 7

Output:

```text
inside outer: changed outer
module: changed global
```

`nonlocal x` modifies `outer()`'s `x`. `global x` modifies the module-level `x`.

---

## Problem 8: Counter Closure

Complete the function so that each call to the returned function increments and returns the count.

In [10]:
def make_counter():
    count = 0

    def counter():
        # Your code here
        return count

    return counter

# c = make_counter()
# print(c())
# print(c())
# print(c())

### Solution 8

In [11]:
def make_counter():
    count = 0

    def counter():
        nonlocal count
        count += 1
        return count

    return counter

c = make_counter()
print(c())
print(c())
print(c())

1
2
3


Expected output:

```text
1
2
3
```

`count += 1` is a rebinding operation, so `nonlocal count` is required.

---

## Problem 9: Independent Closures

Predict the output.

In [12]:
def make_counter(start):
    count = start

    def counter():
        nonlocal count
        count += 1
        return count

    return counter

a = make_counter(0)
b = make_counter(10)

print(a())
print(a())
print(b())
print(a())
print(b())

1
2
11
3
12


### Solution 9

Output:

```text
1
2
11
3
12
```

Each call to `make_counter()` creates a separate enclosing scope. Therefore, `a` and `b` have independent `count` variables.

---

## Problem 10: `nonlocal` with Mutable Objects

Does this code need `nonlocal`? Predict the output.

In [13]:
def outer():
    values = []

    def inner():
        values.append('A')
        values.append('B')

    inner()
    print(values)

outer()

['A', 'B']


### Solution 10

Output:

```text
['A', 'B']
```

No `nonlocal` is needed because `inner()` mutates the list object. It does not rebind the name `values`.

---

## Problem 11: Rebinding a Mutable Object

Predict the output.

In [14]:
def outer():
    values = ['outer']

    def inner():
        values = ['inner']
        values.append('local only')
        print('inside:', values)

    inner()
    print('outside:', values)

outer()

inside: ['inner', 'local only']
outside: ['outer']


### Solution 11

Output:

```text
inside: ['inner', 'local only']
outside: ['outer']
```

`values = ['inner']` creates a new local variable in `inner()`. It does not replace the `values` list from `outer()`.

---

## Problem 12: Fix Rebinding with `nonlocal`

Fix the previous problem so `inner()` replaces the enclosing `values` list.

In [15]:
def outer():
    values = ['outer']

    def inner():
        # Your fix here
        values = ['inner']
        values.append('replaced')

    inner()
    print(values)

# outer()

### Solution 12

In [16]:
def outer():
    values = ['outer']

    def inner():
        nonlocal values
        values = ['inner']
        values.append('replaced')

    inner()
    print(values)

outer()

['inner', 'replaced']


Expected output:

```text
['inner', 'replaced']
```

---

## Problem 13: Compile-Time Scope with `nonlocal`

Predict what happens.

In [17]:
def outer():
    x = 10

    def inner():
        print(x)
        x = 20

    try:
        inner()
    except Exception as ex:
        print(type(ex).__name__ + ':', ex)

outer()

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value


### Solution 13

This raises `UnboundLocalError`.

Because `x = 20` appears inside `inner()`, Python treats `x` as local to `inner()` throughout the function. Therefore, `print(x)` tries to read the local `x` before assignment.

The fix is:

In [18]:
def outer():
    x = 10

    def inner():
        nonlocal x
        print(x)
        x = 20

    inner()
    print(x)

outer()

10
20


Expected output:

```text
10
20
```

---

## Problem 14: Function Factory with Private State

Implement `make_accumulator()` so that it remembers a running total.

In [19]:
def make_accumulator():
    total = 0

    def add(value):
        # Your code here
        return total

    return add

# acc = make_accumulator()
# print(acc(5))
# print(acc(10))
# print(acc(-3))

### Solution 14

In [20]:
def make_accumulator():
    total = 0

    def add(value):
        nonlocal total
        total += value
        return total

    return add

acc = make_accumulator()
print(acc(5))
print(acc(10))
print(acc(-3))

5
15
12


Expected output:

```text
5
15
12
```

`total += value` rebinds `total`, so `nonlocal total` is required.

---

## Problem 15: Comprehensive Challenge

Predict the exact output.

In [21]:
x = 'global'

def outer():
    x = 'outer'
    log = []

    def middle():
        nonlocal x
        x = 'middle changed outer'
        y = 'middle local'

        def inner():
            nonlocal y
            global x
            y = 'inner changed middle'
            x = 'inner changed global'
            log.append(y)

        inner()
        log.append(x)
        log.append(y)

    middle()
    print('outer x:', x)
    print('log:', log)

outer()
print('global x:', x)

outer x: middle changed outer
log: ['inner changed middle', 'middle changed outer', 'inner changed middle']
global x: inner changed global


### Solution 15

Output:

```text
outer x: middle changed outer
log: ['inner changed middle', 'middle changed outer', 'inner changed middle']
global x: inner changed global
```

Explanation:

- `middle()` declares `nonlocal x`, so its `x` is `outer()`'s `x`.
- `inner()` declares `global x`, so its `x` is the module-level `x`, not `outer()`'s `x`.
- `inner()` declares `nonlocal y`, so it modifies `middle()`'s `y`.
- `log` is mutated, not rebound, so no `nonlocal log` is required.

## Key Takeaways

- Inner functions can read names from enclosing function scopes.
- Assigning to a name inside a function makes it local unless declared `global` or `nonlocal`.
- `nonlocal` rebinding targets the nearest enclosing function scope where that name exists.
- `nonlocal` cannot target module-level global names.
- `global` targets the module scope, even from deeply nested functions.
- Mutating an enclosing mutable object does not require `nonlocal`.
- Rebinding an enclosing variable requires `nonlocal`.
- Closures allow functions to remember state after the outer function has finished executing.
- Prefer closures for controlled private state, but avoid unnecessary hidden state when explicit parameters are clearer.